In [75]:
import os
import math
import numpy as np
import pandas as pd
# Constant for number of faces. This won't change unless dramatic changes are made to GCHP.
NUM_FACES = 6

In [76]:
def autoupdate_nx_ny(num_nodes, num_cores_per_node):
    """Automatically calculate NX and NY based on total cores."""
    total_cores = num_nodes * num_cores_per_node
    Z = total_cores // NUM_FACES
    N = math.ceil(math.sqrt(Z))

    while N > 0:
        if Z % N == 0:
            NX = N
            NY = Z // N
            return NX, NY
        N -= 1

    raise ValueError("Failed to compute NX and NY")

In [77]:
def verify_nx_ny(nx, ny, total_cores, cs_res):
    """Validate NX and NY settings based on total cores and resolution."""
    errors = []
    warnings = []

    if nx * ny * NUM_FACES != total_cores:
        errors.append("ERROR: NX*NY*NUM_FACES does not match total cores.")

    if cs_res % 2 != 0:
        errors.append("ERROR: CS_RES must be an even number.")

    if cs_res // nx < 4 or cs_res // ny < 4:
        errors.append("ERROR: Subdomain size too small. Each must be ≥ 4.")

    if nx // ny * 2 >= 5 or (ny // nx) * 2 >= 5:
        warnings.append("WARNING: NX x NY has a side ratio ≥ 2.5. Consider balancing.")

    return errors, warnings

In [78]:
NUM_CORES = 576
NUM_NODES = 2
NUM_CORES_PER_NODE = NUM_CORES // NUM_NODES
CS_RES = 48

NX, NY = autoupdate_nx_ny(NUM_NODES, NUM_CORES_PER_NODE)
errors, warnings = verify_nx_ny(NX, NY, NUM_NODES * NUM_CORES_PER_NODE, CS_RES)

print(f"NX = {NX}, NY = {NY}")
for e in errors:
    print(e)
for w in warnings:
    print(w)

NX = 8, NY = 12


In [79]:
def reshape_og_assignment(df: pd.DataFrame) -> np.ndarray:
    """Restructure the DataFrame into NUM_FACES * RES * RES shape."""
    return df.values.reshape((NUM_FACES, CS_RES, CS_RES))

df = pd.read_csv(f"test/og_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv", index_col=0)
reshaped = reshape_og_assignment(df)
# Write the reshaped data to a file for verification
os.makedirs(f"test/reshaped_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}", exist_ok=True)
# Repeat each face
for i in range(NUM_FACES):
    np.savetxt(f"test/reshaped_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}/face_{i}.txt", reshaped[i], fmt='%d')

In [80]:
def start_end_og_assignment(
    reshaped: np.ndarray
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate start and end indices for each core in each face in the original assignment."""
    x_start_ends = []
    y_start_ends = []
    num_faces, res_y, res_x = reshaped.shape
    for face in range(num_faces):
        face_data = reshaped[face]
        core_ids = np.unique(face_data)
        for core_id in core_ids:
            ys, xs = np.where(face_data == core_id)
            x_start = xs.min()
            x_end = xs.max() + 1  # end is exclusive
            y_start = ys.min()
            y_end = ys.max() + 1  # end is exclusive
            x_start_ends.append([face, core_id, x_start, x_end])
            y_start_ends.append([face, core_id, y_start, y_end])
    x_start_df = pd.DataFrame(x_start_ends, columns=['face', 'core_id', 'x_start', 'x_end'])
    y_start_df = pd.DataFrame(y_start_ends, columns=['face', 'core_id', 'y_start', 'y_end'])
    return x_start_df, y_start_df


x_start_df, y_start_df = start_end_og_assignment(reshaped)
x_start_df.to_csv(f"test/reshaped_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}/x_start_ends.csv", index=False)
y_start_df.to_csv(f"test/reshaped_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}/y_start_ends.csv", index=False)

In [81]:
def x_round(x):
    """Ceil the index if it is in the first half of CS_RES, else floor it."""
    if x < CS_RES / 2:
        return math.ceil(x)
    else:
        return math.floor(x)

def nx_ny_to_reshaped_assignment(nx, ny):
    """Generate reshaped assignment based on NX and NY, with y split using chunking strategy."""
    reshaped = np.zeros((NUM_FACES, CS_RES, CS_RES), dtype=int)
    core_id = 0
    # Calculate y chunk sizes
    half_res = CS_RES // 2
    half_ny = ny // 2
    base = half_res // half_ny
    remainder = half_res % half_ny
    half_chunks = [base + 1 if i < remainder else base for i in range(half_ny)]
    mirror_half_chunks = half_chunks[::-1]
    full_chunks = half_chunks + mirror_half_chunks
    print(f"Full chunks for y split: {full_chunks}")
    for face in range(NUM_FACES):
        y_end = 0
        for i in range(ny):
            y_start = y_end
            y_end += full_chunks[i]
            for j in range(nx):
                x_start = x_round(j * (CS_RES / nx))
                x_end = x_round((j + 1) * (CS_RES / nx)) if j < nx - 1 else CS_RES

                # Assign the core ID to the subdomain
                if face == 0:
                    print(f"Face {face}, Core ID {core_id}: x_start={x_start}, x_end={x_end}, y_start={y_start}, y_end={y_end}")
                reshaped[face, y_start:y_end, x_start:x_end] = core_id
                core_id += 1
    return reshaped

generated_reshaped = nx_ny_to_reshaped_assignment(NX, NY)
# Write the generated reshaped data to a file for verification
os.makedirs(f"test/generated_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}", exist_ok=True)
# Repeat each face
for i in range(NUM_FACES):
    np.savetxt(f"test/generated_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}/face_{i}.txt", generated_reshaped[i], fmt='%d')

Full chunks for y split: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Face 0, Core ID 0: x_start=0, x_end=6, y_start=0, y_end=4
Face 0, Core ID 1: x_start=6, x_end=12, y_start=0, y_end=4
Face 0, Core ID 2: x_start=12, x_end=18, y_start=0, y_end=4
Face 0, Core ID 3: x_start=18, x_end=24, y_start=0, y_end=4
Face 0, Core ID 4: x_start=24, x_end=30, y_start=0, y_end=4
Face 0, Core ID 5: x_start=30, x_end=36, y_start=0, y_end=4
Face 0, Core ID 6: x_start=36, x_end=42, y_start=0, y_end=4
Face 0, Core ID 7: x_start=42, x_end=48, y_start=0, y_end=4
Face 0, Core ID 8: x_start=0, x_end=6, y_start=4, y_end=8
Face 0, Core ID 9: x_start=6, x_end=12, y_start=4, y_end=8
Face 0, Core ID 10: x_start=12, x_end=18, y_start=4, y_end=8
Face 0, Core ID 11: x_start=18, x_end=24, y_start=4, y_end=8
Face 0, Core ID 12: x_start=24, x_end=30, y_start=4, y_end=8
Face 0, Core ID 13: x_start=30, x_end=36, y_start=4, y_end=8
Face 0, Core ID 14: x_start=36, x_end=42, y_start=4, y_end=8
Face 0, Core ID 15: x_start=42, x_end=4

In [82]:
def compare_assignments(reshaped, generated):
    """Compare the original reshaped assignment with the generated one."""
    for face in range(NUM_FACES):
        if not np.array_equal(reshaped[face], generated[face]):
            print(f"Mismatch found in face {face}")
            # Find the first mismatch
            mismatch_indices = np.where(reshaped[face] != generated[face])
            print(f"Number of mismatches in face {face}: {len(mismatch_indices[0])}")
            print(f"Mismatch at indices: {list(zip(*mismatch_indices))}")
            # Print the mismatched values
            print(f"Original values: {reshaped[face][mismatch_indices]}")
            print(f"Generated values: {generated[face][mismatch_indices]}")
            return False
    print("All faces match.")
    return True


# Compare the original reshaped assignment with the generated one
comparison_result = compare_assignments(reshaped, generated_reshaped)

All faces match.


In [74]:
def reshape_generated_assignment(generated):
    """Reshape the generated assignment back to original format."""
    reshaped_df = pd.DataFrame(generated.reshape(NUM_FACES * CS_RES * CS_RES))
    reshaped_df.index.name = 'Column'
    reshaped_df.columns = ['Rank']
    return reshaped_df
reshaped_df = reshape_generated_assignment(generated_reshaped)


# Compare the reshaped DataFrame with the original one if it exists
def verify_and_export(
    reshaped_df,
    generated_path,
    original_path = None
):
    """Verify reshaped DataFrame against the original and export if necessary."""
    if os.path.exists(original_path):
        original_df = pd.read_csv(original_path, index_col=0)
        if not np.array_equal(original_df.values, reshaped_df.values):
            print("Mismatch found in reshaped DataFrame.")
            return False
        else:
            print("Reshaped DataFrame matches the original one.")
    else:
        print("Original file not found. Exporting reshaped DataFrame.")

    # Export if
    os.makedirs(os.path.dirname(generated_path), exist_ok=True)
    reshaped_df.to_csv(generated_path)
    return True

# Call the function with appropriate paths
verify_and_export(
    reshaped_df,
    f"test/generated_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv",
    f"test/og_assignments/c{CS_RES}_p{NUM_NODES*NUM_CORES_PER_NODE}.csv"
)

Reshaped DataFrame matches the original one.


True